# Training a model, for real — the houses

**Darija AI · episode 10**

For nine episodes we drew it. Now we write it. Everything here is something
you have already seen on screen:

| Here | Episode |
|---|---|
| the houses, and the model `price = a · surface + b` | ep01 |
| the Loss | ep02 |
| the gradient | ep03, ep05 |
| the data | ep08 |
| the train / test split | ep09 |

Run each cell with **Shift + Enter**.

In [ ]:
# @title Setup — run me first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# The same data as the videos. Read locally if the repo is cloned, else from GitHub.
DATA = "data/" if Path("data/houses.csv").exists() else \
       "https://raw.githubusercontent.com/Dahimi/darija-ai-notebooks/main/data/"

# The channel's colours: blue = data, purple = model, red = error.
BLUE, PURPLE, RED, GREEN, INK = "#58C4DD", "#B18AE0", "#FF6B6B", "#7BD389", "#E9EDF2"
plt.style.use("dark_background")
plt.rcParams.update({
    "figure.facecolor": "#0E1117", "axes.facecolor": "#0E1117",
    "axes.edgecolor": "#39404E", "grid.color": "#39404E", "axes.grid": True,
    "figure.figsize": (7, 4.5), "font.size": 12, "axes.spines.top": False,
    "axes.spines.right": False,
})

## 1. The data

Eight houses: a surface in m², and a price in *milion*.

In [ ]:
houses = pd.read_csv(DATA + "houses.csv")
houses

In [ ]:
x = houses["surface_m2"].values
y = houses["price_milion"].values

plt.scatter(x, y, color=BLUE, s=60)
plt.xlabel("surface (m²)")
plt.ylabel("price (milion)")
plt.show()

## 2. The model

A straight line, with two knobs: `a` and `b`.

    price = a · surface + b

Let's try a guess.

In [ ]:
def predict(x, a, b):
    return a * x + b

a, b = 0.45, 6.0

plt.scatter(x, y, color=BLUE, s=60)
plt.plot(x, predict(x, a, b), color=PURPLE, linewidth=3)
plt.xlabel("surface (m²)")
plt.ylabel("price (milion)")
plt.show()

## 3. The Loss

How wrong is the guess? For each house: the gap between what we predicted and
the real price, squared. Then add them all up.

In [ ]:
def loss(a, b):
    errors = predict(x, a, b) - y
    return np.sum(errors ** 2)

loss(0.45, 6.0)

The walk from episode 2: keep `b = 3.8` and turn the `a` knob.

In [ ]:
for a in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    print(f"a = {a:.2f}   Loss = {loss(a, 3.8):7.1f}")

## 4. The gradient

The gradient tells us, for each knob, which way the Loss goes up — and how
steeply. We go the other way.

    dLoss/da = 2 · average( error · surface )
    dLoss/db = 2 · average( error )

We take the **average** over the houses rather than the sum. The direction is
exactly the same; only the size changes, and this way the same step size works
whether we have 8 houses or 8,000.

In [ ]:
def gradient(x, y, a, b):
    errors = predict(x, a, b) - y
    grad_a = 2 * np.mean(errors * x)
    grad_b = 2 * np.mean(errors)
    return grad_a, grad_b

## 5. The training loop

Start with the knobs at zero. Then, again and again: compute the gradient,
take a small step downhill.

In [ ]:
def train(x, y, learning_rate=0.0001, steps=500_000):
    a, b = 0.0, 0.0
    for step in range(steps):
        grad_a, grad_b = gradient(x, y, a, b)
        a = a - learning_rate * grad_a
        b = b - learning_rate * grad_b

        if step in (0, 1, 2, 3, 10, 100, 1_000, 10_000, 100_000, steps - 1):
            loss = np.mean((predict(x, a, b) - y) ** 2)
            print(f"step {step:>7,}   a = {a:.3f}   b = {b:6.3f}   loss = {loss:8.2f}")
    return a, b

a, b = train(x, y)

In [ ]:
plt.scatter(x, y, color=BLUE, s=60)
plt.plot(x, predict(x, a, b), color=PURPLE, linewidth=3)
plt.xlabel("surface (m²)")
plt.ylabel("price (milion)")
plt.title(f"price = {a:.2f} · surface + {b:.2f}")
plt.show()

The model is trained. Let's use it: how much for a 60 m² house?

In [ ]:
predict(60, a, b)

## 6. More houses, and the honest test

Eight houses is a toy. Here are 200, sampled from a whole city.

Before training, we put some aside — the **test set**. The model never sees
them during training. They are how we find out if it really learned.

In [ ]:
city = pd.read_csv(DATA + "city_sample.csv")

shuffled = city.sample(frac=1, random_state=42)   # shuffle the rows
train_set = shuffled.iloc[:160]                   # 80% to learn from
test_set = shuffled.iloc[160:]                    # 20% kept in the envelope

plt.scatter(train_set["surface_m2"], train_set["price_milion"], color=BLUE, s=20, label="train")
plt.scatter(test_set["surface_m2"], test_set["price_milion"], color=RED, s=20, label="test")
plt.xlabel("surface (m²)")
plt.ylabel("price (milion)")
plt.legend()
plt.show()

In [ ]:
a, b = train(train_set["surface_m2"].values, train_set["price_milion"].values)

How far off are we, on average, in *milion*? On the houses it trained on,
and on the houses it has never seen.

In [ ]:
def average_miss(data, a, b):
    return np.mean(np.abs(predict(data["surface_m2"], a, b) - data["price_milion"]))

print(f"train: {average_miss(train_set, a, b):.1f} milion")
print(f"test:  {average_miss(test_set, a, b):.1f} milion")

## 7. The whole city

In real life you never get to see the whole population. Our city is invented,
so here we can cheat — and check whether the test set told the truth.

In [ ]:
full_city = pd.read_csv(DATA + "city_full.csv")
print(f"the whole city ({len(full_city):,} houses): {average_miss(full_city, a, b):.1f} milion")

## 8. The library

Everything we just wrote, a library like scikit-learn does in one line.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(train_set[["surface_m2"]], train_set["price_milion"])

print(f"ours:         a = {a:.3f}   b = {b:.3f}")
print(f"scikit-learn: a = {model.coef_[0]:.3f}   b = {model.intercept_:.3f}")

Same answer. The library is not magic — it is the loop above, made fast.

## 9. One more column

About 10 milion off is a lot. Episode 8: a column the model doesn't have is
something it cannot learn. Our data has one more — whether the house is in an
expensive neighbourhood (`ghalya`).

In [ ]:
columns = ["surface_m2", "ghalya"]

model = LinearRegression()
model.fit(train_set[columns], train_set["price_milion"])

def average_miss_2(data):
    return np.mean(np.abs(model.predict(data[columns]) - data["price_milion"]))

print(f"test:       {average_miss_2(test_set):.1f} milion")
print(f"whole city: {average_miss_2(full_city):.1f} milion")

## Try it yourself

- Change `learning_rate` to `0.0002`. What happens to the loss? Why?
- Train with only `steps=100`. How close do `a` and `b` get?
- Put your own surface into `predict(...)`.
- In section 6, change `random_state=42` to another number. Do the train and
  test errors move?